# Multi-LLM-as-Judge: ADRD Medical AI
## Complete SMU SuperPOD Experiment Notebook

**Tailored to:** `mkotha` on SMU SuperPOD | Account: `xnluo_ai_chatbot_cognitive_0002`

| Phase | Cell | What happens |
|---|---|---|
| 0 | — | SSH + SOCKS proxy + tmux + srun (local terminal) |
| 1 | 1 | Verify GPU node, venv, paths |
| 1b | 1b | ⬇️ Fetch source datasets from committed CSVs (no HF/Kaggle needed) |
| 2 | 2 | Install deps |
| 3 | 3 | Load .env + HF cache |
| 3 | 4 | Download all 4 judge models |
| 4 | 5 | Kill any old vLLM + launch 4 fresh servers |
| 4 | 6 | Tail logs (optional) |
| 5 | 7 | Health check all 4 judges |
| 6 | 8 | Repo structure check |
| 6 | 9 | Unit tests |
| 7 | 10 | Exp 1 — Dataset analysis |
| 8 | 11 | Exp 2 — Agreement analysis |
| 9 | 12 | Exp 3 — Rubric sensitivity |
| 10 | 13 | Exp 4 — Box plots |
| 11 | 14 | Results summary + CSV |
| 12 | 15 | Display figures inline |
| 13 | 16 | Push to GitHub |
| 14 | 17 | Cleanup / kill vLLM |

---

## Phase 0 — Terminal Commands (run BEFORE opening this notebook)

```bash
# 1. On your LOCAL machine:
ssh -C -D 8080 mkotha@superpod.smu.edu

# 2. On SuperPOD login node:
tmux new -s cs7325
# (to reattach later: tmux attach -t cs7325)

# 3. Request GPU node (inside tmux):
srun -A xnluo_ai_chatbot_cognitive_0002 -N1 -G2 -c32 --mem=192G --time=4:00:00 --pty $SHELL

# 4. On the GPU compute node:
cd /lustre/smuexa01/client/users/mkotha/CS7325

# 5. Clone repo (only first time):
git clone https://YOUR_GITHUB_TOKEN@github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI.git
cd Multi_LLM_as_Judge_Medical_AI

# 6. Create .env file (only first time):
cat > .env << 'EOF'
HF_TOKEN=hf_xxxxxxxxxxxx
GITHUB_TOKEN=ghp_xxxxxxxxxxxx
LOCAL_VLLM_JUDGE1_URL=http://localhost:8001
LOCAL_VLLM_JUDGE2_URL=http://localhost:8002
LOCAL_VLLM_JUDGE3_URL=http://localhost:8003
LOCAL_VLLM_JUDGE4_URL=http://localhost:8004
EOF

# 7. Activate venv and launch JupyterLab:
source /lustre/smuexa01/client/users/mkotha/CS7325/.venv/bin/activate
cd /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI
jupyter lab --ip=0.0.0.0 --no-browser
# Open the printed URL in your browser (SOCKS proxy port 8080)
```


---
## Phase 1 — Verify GPU Node & Environment

In [ ]:
# CELL 1: Verify GPU, venv, paths
import subprocess, os, sys, importlib
from pathlib import Path
from datetime import datetime

def shell(cmd, check=False):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (result.stdout + result.stderr).strip()
    if out: print(out)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return result.returncode

LUSTRE_BASE = Path('/lustre/smuexa01/client/users/mkotha/CS7325')
ROOT        = LUSTRE_BASE / 'Multi_LLM_as_Judge_Medical_AI'
VENV        = LUSTRE_BASE / '.venv'
HF_CACHE    = LUSTRE_BASE / 'hf_models'
LOG_DIR     = LUSTRE_BASE / 'vllm_logs'
PYTHON      = str(VENV / 'bin' / 'python')
SOURCE_DIR  = ROOT / 'benchmark_dataset' / 'source_datasets'

SAMPLE_N = 100  # rows per source dataset

HF_CACHE.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Repo root  : {ROOT}')
print(f'Source dir : {SOURCE_DIR}')
print(f'SAMPLE_N   : {SAMPLE_N} rows per dataset')
print(f'Python     : {sys.executable}')
print(f'Started    : {datetime.now().isoformat()}')

print('\n=== GPU Status ===')
shell('nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader')
print('\n=== SLURM ===')
shell('echo "Node: $SLURMD_NODENAME  Job: $SLURM_JOB_ID  GPUs: $SLURM_GPUS  CPUs: $SLURM_CPUS_ON_NODE"')
venv_ok = str(VENV) in sys.executable
print(f'venv: {"✅ active" if venv_ok else "❌ NOT active — run: source " + str(VENV) + "/bin/activate"}')
shell(f'df -h {LUSTRE_BASE} | tail -1')

---
## Phase 1b — Fetch Source Datasets

Fetches datasets directly from committed CSVs in the sibling repo:
`github.com/m22oct2000/Multi-LLMs-as-Judge/tree/main/dataset`

**No HuggingFace Hub, no Kaggle, no API tokens required.**

| # | Output file | Source CSV | Rows |
|---|---|---|---|
| 1 | `health_advice.csv` | `medical_meadow_health_advice/medical_meadow_health_advice.csv` | 100 |
| 2 | `medquad.csv` | `MedQuAD/train.csv` | 100 |
| 3 | `meddialog.csv` | `MedDialog/validation.csv` | 100 |
| 4 | `curated_seed.csv` | `sample.csv` (22 expert Q&A rows) | all |

> Skips any file that already exists. Re-run to refresh.

In [ ]:
# CELL 1b: Fetch source datasets from committed CSVs in Multi-LLMs-as-Judge repo.
# Runs build_source_datasets.py which uses raw.githubusercontent.com — no HF/Kaggle needed.

build_script = ROOT / 'benchmark_dataset' / 'build_source_datasets.py'
print(f'Running: {build_script}')
rc = shell(f'{PYTHON} {build_script} --n {SAMPLE_N}')
if rc != 0:
    raise RuntimeError('build_source_datasets.py failed — check output above')

import pandas as pd
print('\n=== Source dataset files ===')
total = 0
for f in sorted(SOURCE_DIR.glob('*.csv')):
    n = len(pd.read_csv(f))
    total += n
    print(f'  ✅ {f.name:35s} {n:>4} rows  ({f.stat().st_size // 1024} KB)')
print(f'\n  Total rows: {total}')
print(f'  After stratified sampling: ~{20*5} rows/dataset × 3 datasets + curated = ~{20*5*3+22} total')

---
## Phase 2 — Install Dependencies

In [ ]:
# CELL 2: Install all deps
REQUIREMENTS = ROOT / 'requirements.txt'
shell(f'{sys.executable} -m pip install -r {REQUIREMENTS} -q')
shell(f'{sys.executable} -m pip install vllm -q')
for pkg in ['tabulate', 'ipywidgets']:
    shell(f'{sys.executable} -m pip install {pkg} -q')
REQUIRED = ['vllm', 'httpx', 'datasets', 'huggingface_hub', 'pandas',
            'matplotlib', 'seaborn', 'dotenv', 'pytest', 'tabulate']
all_ok = True
for pkg in REQUIRED:
    try:
        m = importlib.import_module(pkg.replace('-', '_'))
        print(f'  ✅ {pkg:<20} {getattr(m, "__version__", "?")}'  )
    except ImportError:
        print(f'  ❌ {pkg}'); all_ok = False
print('✅ All OK' if all_ok else '❌ Some missing — re-run')

---
## Phase 3 — Download Judge Models

| Judge | Model | GPU | Port |
|---|---|---|---|
| medgemma  | `google/medgemma-4b-it`    | GPU 0 | 8001 |
| biomistral | `BioMistral/BioMistral-7B` | GPU 1 | 8002 |
| meditron  | `epfl-llm/meditron-7b`     | GPU 0 | 8003 |
| medalpaca | `medalpaca/medalpaca-7b`   | GPU 1 | 8004 |


In [ ]:
# CELL 3: Load .env + set HF cache
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
os.environ['HF_HOME']            = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE']  = str(HF_CACHE / 'datasets')
HF_TOKEN = os.environ.get('HF_TOKEN')
GH_TOKEN = os.environ.get('GITHUB_TOKEN')
print(f'HF_TOKEN : {"✅ set" if HF_TOKEN else "❌ NOT SET"}')
print(f'GH_TOKEN : {"✅ set" if GH_TOKEN else "⚠️  not set"}')
shell(f'du -sh {HF_CACHE} 2>/dev/null || echo empty')

In [ ]:
# CELL 4: Download judge models (skips already-cached)
from huggingface_hub import snapshot_download
import time
JUDGE_MODELS = [
    {'id': 'medgemma',   'hf_id': 'google/medgemma-4b-it',    'needs_token': True},
    {'id': 'biomistral', 'hf_id': 'BioMistral/BioMistral-7B', 'needs_token': False},
    {'id': 'meditron',   'hf_id': 'epfl-llm/meditron-7b',     'needs_token': False},
    {'id': 'medalpaca',  'hf_id': 'medalpaca/medalpaca-7b',    'needs_token': False},
]
for m in JUDGE_MODELS:
    md = HF_CACHE / f'models--{m["hf_id"].replace("/", "--")}'
    if md.exists() and any(md.rglob('config.json')):
        print(f'✅ [{m["id"]}] cached'); continue
    if m['needs_token'] and not HF_TOKEN:
        print(f'❌ [{m["id"]}] needs HF_TOKEN'); continue
    t0 = time.time()
    try:
        snapshot_download(repo_id=m['hf_id'], cache_dir=str(HF_CACHE),
            token=HF_TOKEN if m['needs_token'] else None,
            ignore_patterns=['*.msgpack','*.h5','flax_model*','tf_model*','rust_model*'])
        print(f'✅ [{m["id"]}] done in {(time.time()-t0)/60:.1f} min')
    except Exception as e:
        print(f'❌ [{m["id"]}] {e}')
shell(f'du -sh {HF_CACHE}')

---
## Phase 4 — Launch vLLM Servers

| Judge | GPU | Port | GPU Mem |
|---|---|---|---|
| medgemma  | GPU 0 | 8001 | 20% |
| biomistral | GPU 1 | 8002 | 50% |
| meditron  | GPU 0 | 8003 | 65% |
| medalpaca | GPU 1 | 8004 | 42% |


In [ ]:
# CELL 5: Kill old vLLM + launch 4 fresh servers
import time, subprocess
LOG_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
time.sleep(3)
subprocess.run('kill -9 $(fuser /dev/nvidia0 /dev/nvidia1 2>/dev/null) 2>/dev/null || true', shell=True)
time.sleep(8)
subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)

def get_snapshot_path(hf_id):
    snap_dir = HF_CACHE / ('models--' + hf_id.replace('/', '--')) / 'snapshots'
    if not snap_dir.exists(): raise FileNotFoundError(f'Not cached: {hf_id}')
    snaps = list(snap_dir.iterdir())
    if not snaps: raise FileNotFoundError(f'Empty snapshots for {hf_id}')
    return str(sorted(snaps, key=lambda p: p.stat().st_mtime)[-1])

JUDGE_CONFIGS = [
    {'id':'medgemma',  'hf_id':'google/medgemma-4b-it',    'port':8001,'gpu':'0','max_len':4096,'gpu_mem':0.20,'extra':'--trust-remote-code'},
    {'id':'biomistral','hf_id':'BioMistral/BioMistral-7B', 'port':8002,'gpu':'1','max_len':4096,'gpu_mem':0.50,'extra':''},
    {'id':'meditron',  'hf_id':'epfl-llm/meditron-7b',     'port':8003,'gpu':'0','max_len':2048,'gpu_mem':0.65,'extra':''},
    {'id':'medalpaca', 'hf_id':'medalpaca/medalpaca-7b',   'port':8004,'gpu':'1','max_len':2048,'gpu_mem':0.42,'extra':''},
]
snapshot_paths = {}; all_cached = True
for j in JUDGE_CONFIGS:
    try:
        snapshot_paths[j['id']] = get_snapshot_path(j['hf_id'])
        print(f"✅ [{j['id']}] cached")
    except FileNotFoundError as e:
        print(f"❌ [{j['id']}] {e}"); all_cached = False
if not all_cached: raise RuntimeError('Run Cell 4 first.')

def wait_healthy(port, name, timeout=600):
    import urllib.request
    t0 = time.time()
    while time.time()-t0 < timeout:
        try:
            with urllib.request.urlopen(f'http://localhost:{port}/health', timeout=5) as r:
                if r.status == 200: return True
        except: pass
        time.sleep(10); print(f'  ⏳ [{name}] {time.time()-t0:.0f}s', end='\r')
    return False

def launch_judge(j):
    log = str(LOG_DIR / f'vllm_{j["id"]}.log')
    subprocess.run(f'fuser -k {j["port"]}/tcp 2>/dev/null || true', shell=True); time.sleep(2)
    cmd = (f'CUDA_VISIBLE_DEVICES={j["gpu"]} HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 HF_HOME={HF_CACHE} '
           f'{PYTHON} -m vllm.entrypoints.openai.api_server '
           f'--model {snapshot_paths[j["id"]]} --port {j["port"]} --host 0.0.0.0 '
           f'--gpu-memory-utilization {j["gpu_mem"]} --max-model-len {j["max_len"]} '
           f'--dtype bfloat16 {j["extra"]} > {log} 2>&1')
    proc = subprocess.Popen(cmd, shell=True)
    print(f'🚀 [{j["id"]}] PID={proc.pid} GPU={j["gpu"]} port={j["port"]}')
    if wait_healthy(j['port'], j['id']):
        print(f'✅ [{j["id"]}] UP on :{j["port"]}')
    else:
        subprocess.run(f'tail -15 {log}', shell=True)
        raise RuntimeError(f"{j['id']} failed. Fix then re-run.")
    return proc

procs = {}
for j in JUDGE_CONFIGS:
    print(f'\n--- {j["id"]} GPU{j["gpu"]} :{j["port"]} ---')
    procs[j['id']] = launch_judge(j)
JUDGE_URLS = {j['id']: f'http://localhost:{j["port"]}' for j in JUDGE_CONFIGS}
print('\n✅ ALL 4 JUDGES LAUNCHED')

In [ ]:
# CELL 6 (optional): tail vLLM logs
if 'JUDGE_CONFIGS' not in dir():
    JUDGE_CONFIGS = [{'id':'medgemma','gpu':'0','port':8001},{'id':'biomistral','gpu':'1','port':8002},
                     {'id':'meditron','gpu':'0','port':8003},{'id':'medalpaca','gpu':'1','port':8004}]
for j in JUDGE_CONFIGS:
    print(f'\n--- [{j["id"]}] ---')
    shell(f'tail -8 {LOG_DIR}/vllm_{j["id"]}.log 2>/dev/null || echo "(no log)"')

---
## Phase 5 — Health Check

In [ ]:
# CELL 7: Poll all 4 vLLM servers — 20 min max
import httpx, time
if 'JUDGE_URLS' not in dir():
    JUDGE_URLS = {'medgemma':'http://localhost:8001','biomistral':'http://localhost:8002',
                  'meditron':'http://localhost:8003','medalpaca':'http://localhost:8004'}
MAX_WAIT=1200; POLL=30; ready={k:False for k in JUDGE_URLS}; t0=time.time()
while not all(ready.values()):
    elapsed=time.time()-t0
    if elapsed>MAX_WAIT: print('Timeout'); break
    for name,url in JUDGE_URLS.items():
        if ready[name]: continue
        try:
            if httpx.get(f'{url}/health',timeout=4).status_code==200:
                ready[name]=True; print(f'✅ {name} [{elapsed:.0f}s]')
        except: pass
    if not all(ready.values()):
        print(f'⏳ Pending:{[k for k,v in ready.items() if not v]} [{elapsed:.0f}s]'); time.sleep(POLL)
if all(ready.values()): print(f'\n✅ ALL READY in {(time.time()-t0)/60:.1f} min')
shell('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader')

---
## Phase 6 — Pre-Flight Checks

In [ ]:
# CELL 8: Verify repo files
import json
REQUIRED_PATHS = [
    ROOT/'core'/'wrapper.py', ROOT/'core'/'agreement.py',
    ROOT/'core'/'rubric_engine.py', ROOT/'core'/'metrics.py',
    ROOT/'core'/'schemas.py', ROOT/'core'/'consensus_core'/'models.py',
    ROOT/'rubrics'/'rubric1_pemat.json', ROOT/'rubrics'/'rubric2_healthbench.json',
    ROOT/'rubrics'/'rubric3_clinical_eval.json', ROOT/'rubrics'/'rubric4_prometheus.json',
    ROOT/'experiments'/'exp1_dataset_analysis.py', ROOT/'experiments'/'exp2_agreement_analysis.py',
    ROOT/'experiments'/'exp3_rubric_sensitivity.py', ROOT/'experiments'/'exp4_boxplot_agreement.py',
]
all_ok=True
for p in REQUIRED_PATHS:
    ok=p.exists(); print(f'  {"✅" if ok else "❌"} {p.relative_to(ROOT)}')
    if not ok: all_ok=False
print('✅ All present' if all_ok else f'❌ Missing — run: git -C {ROOT} pull origin main')

In [ ]:
# CELL 9: Unit tests
rc = shell(f'{sys.executable} -m pytest {ROOT}/tests/test_core.py -v --tb=short')
print('✅ Passed' if rc==0 else '⚠️  Failed — review before experiments')

---
## Phase 7 — Experiment 1: Dataset Analysis
Stratified benchmark: 20 rows/domain × 5 domains × 3 source datasets + 22 curated = ~322 rows

In [ ]:
# CELL 10: Build benchmark + Exp 1
import importlib, json, pandas as pd
from collections import Counter
os.environ['SAMPLE_N'] = str(SAMPLE_N)

src_files = list(SOURCE_DIR.glob('*.csv'))
print(f'Source CSVs: {[f.name for f in src_files]}')
for f in src_files:
    df_s = pd.read_csv(f); print(f'  {f.name}: {len(df_s)} rows, cols={list(df_s.columns)}')

print('\nBuilding benchmark CSV...')
if shell(f'{sys.executable} {ROOT}/benchmark_dataset/build_agreement_dataset.py') != 0:
    raise RuntimeError('build_agreement_dataset.py failed')
print('Building adrd_questions.json...')
if shell(f'{sys.executable} {ROOT}/benchmark_dataset/build_adrd_questions.py') != 0:
    raise RuntimeError('build_adrd_questions.py failed')

import experiments.exp1_dataset_analysis as exp1_mod; importlib.reload(exp1_mod); exp1_mod.main()

df_bench = pd.read_csv(ROOT/'benchmark_dataset'/'agreement_benchmark.csv')
print(f'\nBenchmark: {len(df_bench)} rows')
if 'domain' in df_bench.columns:
    print(df_bench.groupby('domain')['id'].count().rename('rows').to_string())
if 'source' in df_bench.columns:
    print(df_bench.groupby('source')['id'].count().rename('rows').to_string())
print('✅ Exp1 complete')

---
## Phase 8 — Experiment 2: Per-Rubric Agreement

In [ ]:
# CELL 11: Experiment 2
import experiments.exp2_agreement_analysis as exp2_mod; importlib.reload(exp2_mod); exp2_mod.main()
with open(ROOT/'results'/'exp2_agreement_results.json') as f: exp2_data=json.load(f)
rows=[]
for block in exp2_data:
    res=block['results']; cls=Counter(r['agreement_class'] for r in res); n=len(res)
    rows.append({'Rubric':block['rubric_name'].split('—')[0].strip()[:30],'N':n,
        'Fully%':round(cls.get('fully_agree',0)/n*100,1),
        'Majority%':round(cls.get('majority_agree',0)/n*100,1),
        'Split%':round(cls.get('split',0)/n*100,1),
        'Disagree%':round(cls.get('full_disagree',0)/n*100,1),
        'MeanPW%':round(sum(r['mean_pairwise_agreement'] for r in res)/n,1)})
df_exp2=pd.DataFrame(rows); print(df_exp2.to_markdown(index=False)); print('✅ Exp2 complete')

---
## Phase 9 — Experiment 3: Rubric Sensitivity

In [ ]:
# CELL 12: Experiment 3
import experiments.exp3_rubric_sensitivity as exp3_mod; importlib.reload(exp3_mod); exp3_mod.main()
with open(ROOT/'results'/'exp3_sensitivity_results.json') as f: exp3_data=json.load(f)
rows=[{'Rubric':b['rubric_name'].split('—')[0][:25],'Variant':v['scoring_variant'],
       'Mean%':v['mean_pairwise_agreement'],'Std%':v['std_pairwise_agreement']}
      for b in exp3_data for v in b['variants']]
print(pd.DataFrame(rows).to_markdown(index=False)); print('✅ Exp3 complete')

---
## Phase 10 — Experiment 4: Box Plots

In [ ]:
# CELL 13: Experiment 4
import experiments.exp4_boxplot_agreement as exp4_mod; importlib.reload(exp4_mod); exp4_mod.main()
pngs=sorted((ROOT/'results'/'figures').glob('*.png'))
print(f'✅ {len(pngs)} figures'); [print(f'  📊 {p.name}') for p in pngs]

---
## Phase 11 — Results Summary

In [ ]:
# CELL 14: Summary
print('=== FINAL RESULTS ===')
print(f'Benchmark rows: {len(df_bench)}')
if 'domain' in df_bench.columns:
    for d,n in sorted(df_bench.groupby('domain')['id'].count().items()): print(f'  {d}: {n}')
print(df_exp2.to_markdown(index=False))
for b in exp3_data:
    best=max(b['variants'],key=lambda v:v['mean_pairwise_agreement'])
    print(f'  {b["rubric_name"][:30]} → {best["scoring_variant"]} ({best["mean_pairwise_agreement"]:.1f}%)')
df_exp2.to_csv(ROOT/'results'/'agreement_summary_table.csv',index=False)
print('✅ agreement_summary_table.csv saved')

---
## Phase 12 — Display Figures

In [ ]:
from IPython.display import Image, display, Markdown
for png in sorted((ROOT/'results'/'figures').glob('*.png')):
    display(Markdown(f'### {png.stem}')); display(Image(filename=str(png), width=950))

---
## Phase 13 — Push to GitHub

In [ ]:
# CELL 16: Push results
if GH_TOKEN: shell(f'git -C {ROOT} remote set-url origin https://{GH_TOKEN}@github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI.git')
COMMIT_MSG = f'HPC results (SuperPOD mkotha) [source-100rows-stratified]: {datetime.now().strftime("%Y-%m-%d %H:%M")}'
shell(f'git -C {ROOT} add results/ benchmark_dataset/ notebooks/')
shell(f'git -C {ROOT} commit -m "{COMMIT_MSG}"')
rc = shell(f'git -C {ROOT} push origin main')
print('✅ Pushed!' if rc==0 else f'⚠️  Push failed — try: cd {ROOT} && git push origin main')

---
## Phase 14 — Cleanup

In [ ]:
# CELL 17: Kill vLLM + free GPUs
if 'procs' in dir():
    for name,proc in procs.items():
        try: proc.terminate(); print(f'Terminated [{name}]')
        except: pass
subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
import time; time.sleep(5)
shell('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader')
print('✅ GPUs free. Exit srun: exit')

---
## ✅ All Experiments Complete

| File | Use |
|---|---|
| `results/agreement_summary_table.csv` | Paper Table 2 |
| `results/figures/*.png` | Upload to Overleaf |
| `results/exp2_agreement_results.json` | Full rationales |
| `benchmark_dataset/agreement_benchmark.csv` | Full benchmark |

**Acknowledgement:** *Computational resources provided by SMU O'Donnell Data Science and Research Computing Institute (SuperPOD, 2x A100 80GB, allocation: xnluo_ai_chatbot_cognitive_0002).*
